# Finance Q&A Agent: stress test with smaller models and harder questions

The first evaluation was a **tie on accuracy**: the standard model answered all 13 numeric questions correctly even without code. This notebook tests where the agent's approach should matter more:

| Change | Why |
|---|---|
| **A small model as well as a standard one** | Small, cheap models make more arithmetic slips when answering directly; running code shouldn't care |
| **Minimal reasoning effort** for both approaches | GPT-5-family models reason internally by default, which hides arithmetic weaknesses. The same setting is used for direct and agent, so the comparison stays fair |
| **16 harder questions** | **Distractors** (numbers that aren't needed), **unit conversions** (weekly→monthly, TB→GB, daily→annual) and **multi-step** problems with 4–5 calculations |

Each question is answered four ways: {small, standard} model × {direct answer, agent}. Expected values are computed in Python, not typed by hand.

The agent is the same as in the main notebook (structured reply → every input traced to the question → sandboxed code → retries with feedback), plus four fixes from the first stress-test run:

| Fix | Problem in the first run | Change |
|---|---|---|
| **1. Percentages normalised in code** | Q9: the small model used margin as 0.72 but churn as 3.5, and returned 0.03 instead of 3.25. Every input was real, so the input checks passed | Any input whose source is a percentage reaches the code as a fraction, however the model wrote it. Code that divides a percentage by 100 again is rejected with the reason |
| **2. Tighter clarify rule** | The small model asked "shall I proceed?" on 7 clear questions | Ask only when a number is missing or two readings give different answers; never ask permission or confirm a stated method. Applied to direct and agent alike |
| **3. One worked example** | The small model kept hard-coding numbers in the code (6 rejections) | The prompt shows a complete valid reply. The offline tests check that the example passes the agent's own rules. The direct prompt gets the same example question, so the comparison stays fair |
| **4. Wrong numbers counted separately** | A refusal and a wrong number both scored ✗ | The scorecard splits failures into wrong number, asked to clarify, and rejected, because a wrong number is the dangerous outcome |

**Setup:** add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Colab **Secrets** (🔑) with notebook access on, and check the model names in Step 5.

## Step 1: Install packages

In [1]:
!pip install -q -U langchain-core langchain-openai langchain-anthropic pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 49.4 MB/s eta 0:00:00


## Step 2: The structured reply and the input checks (Upgrades 2 and 3)

The model replies in JSON with either:

- `"status": "answer"` with its **inputs** (name, value, source phrase), **assumptions**, **code** and an explanation; or
- `"status": "clarify"` with a clarifying question, when an answer would mean guessing.

`check_inputs` then verifies every input against the question.

In [2]:
import json
import re
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class Input(BaseModel):
    name: str = Field(pattern=r"^[a-z_][a-z0-9_]*$")
    value: float
    source: str = Field(min_length=1)          # the exact phrase from the question


class AgentReply(BaseModel):
    status: Literal["answer", "clarify"]
    clarifying_question: str | None = None
    assumptions: list[str] = []
    inputs: list[Input] = []
    code: str = ""
    explanation: str = ""


WORD_NUMBERS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
                "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "double": 2, "half": 0.5}
NUMBER_RE = re.compile(r"(\d[\d,]*\.?\d*)\s*(%|(?:k|m|bn|b|thousand|million|billion|percent)\b)?", re.IGNORECASE)
SCALE = {"k": 1e3, "thousand": 1e3, "m": 1e6, "million": 1e6, "b": 1e9, "bn": 1e9, "billion": 1e9}


def normalise(text: str) -> str:
    text = text.lower().replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", text).strip()


def numbers_in(text: str) -> set:
    """Every value a phrase can legitimately stand for: '8%' -> {8, 0.08}, '$1.2M' -> {1.2, 1200000}."""
    values = set()
    for raw, suffix in NUMBER_RE.findall(text.replace("$", "")):
        n = float(raw.replace(",", ""))
        values.add(n)
        s = (suffix or "").lower()
        if s in SCALE:
            values.add(n * SCALE[s])
        if s in ("%", "percent"):
            values.add(n / 100)
    for word, n in WORD_NUMBERS.items():
        if re.search(rf"\b{word}\b", text.lower()):
            values.add(float(n))
    return values


def check_inputs(inputs: list, question: str) -> list:
    """Each input's source phrase must appear in the question, and its value must match a number in that phrase."""
    problems, q = [], normalise(question)
    for inp in inputs:
        if normalise(inp.source) not in q:
            problems.append(f"input '{inp.name}': source phrase \"{inp.source}\" does not appear in the question")
            continue
        if not any(abs(inp.value - v) <= 1e-9 * max(1, abs(v)) for v in numbers_in(inp.source)):
            problems.append(f"input '{inp.name}' = {inp.value} does not match any number in \"{inp.source}\"")
    return problems


def parse_reply(text: str) -> AgentReply:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("no JSON object found in the reply")
    return AgentReply.model_validate_json(match.group(0))

## Step 3: The sandbox: inputs are injected, nothing else is hard-coded

The verified inputs are passed into the sandbox as variables. The code itself may only use those variables, a few structural constants (`0, 1, 2, 12, 100`) and safe built-ins. A hard-coded `10000` in the code is rejected, because it would bypass the input check.

In [3]:
import ast
import contextlib
import io

SAFE_BUILTINS = {"print": print, "round": round, "abs": abs, "min": min, "max": max, "sum": sum,
                 "range": range, "len": len, "int": int, "float": float, "pow": pow}
ALLOWED_CONSTANTS = {0, 1, 2, 12, 100}
BLOCKED_NODES = (ast.Import, ast.ImportFrom, ast.With, ast.While, ast.FunctionDef, ast.AsyncFunctionDef,
                 ast.ClassDef, ast.Lambda, ast.Attribute, ast.Global, ast.Nonlocal, ast.Try, ast.Raise, ast.Delete)


def validate_code(code: str, input_names: set, percent_names: set = frozenset()) -> list:
    try:
        tree = ast.parse(code)
    except SyntaxError as err:
        return [f"syntax error: {err.msg} (line {err.lineno})"]
    assigned = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}
    known = input_names | assigned | set(SAFE_BUILTINS)
    problems = []
    for node in ast.walk(tree):
        line = getattr(node, "lineno", "?")
        if isinstance(node, BLOCKED_NODES):
            problems.append(f"not allowed: {type(node).__name__} (line {line})")
        elif isinstance(node, ast.Call) and not (isinstance(node.func, ast.Name) and node.func.id in SAFE_BUILTINS):
            problems.append(f"not allowed: a call to something other than {sorted(SAFE_BUILTINS)} (line {line})")
        elif isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load) and node.id not in known:
            problems.append(f"unknown variable '{node.id}' (line {line}): declare it as an input")
        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool) \
                and node.value not in ALLOWED_CONSTANTS:
            problems.append(f"hard-coded number {node.value} (line {line}): declare it as an input with its source")
    for node in ast.walk(tree):   # Fix 1: percentages already arrive as fractions
        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Div) and isinstance(node.right, ast.Constant) \
                and node.right.value == 100:
            used = {n.id for n in ast.walk(node.left) if isinstance(n, ast.Name)} & set(percent_names)
            for name in sorted(used):
                problems.append(f"'{name}' is a percentage and already a fraction (e.g. 0.72): don't divide it by 100 (line {node.lineno})")
    if not any(isinstance(n, ast.Call) and getattr(n.func, "id", "") == "print" for n in ast.walk(tree)):
        problems.append("the code must print its answer with print(...)")
    return problems


def run_sandboxed(code: str, inputs: dict, percent_names: set = frozenset()) -> dict:
    problems = validate_code(code, set(inputs), percent_names)
    if problems:
        return {"ok": False, "output": "", "error": "; ".join(problems)}
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(compile(code, "<generated>", "exec"), {"__builtins__": SAFE_BUILTINS, **inputs})   # one namespace, so generators see the inputs
        return {"ok": True, "output": buffer.getvalue().strip(), "error": ""}
    except Exception as err:
        return {"ok": False, "output": buffer.getvalue().strip(), "error": f"{type(err).__name__}: {err}"}


def first_number(text: str):
    match = re.search(r"-?\d[\d,]*\.?\d*", (text or "").replace("$", ""))
    return float(match.group(0).replace(",", "")) if match else None


def text_of(message) -> str:
    """OpenAI returns a string; Anthropic can return a list of content blocks."""
    if isinstance(message.content, str):
        return message.content
    return "".join(b.get("text", "") for b in message.content if isinstance(b, dict))


def _same(a: float, b: float) -> bool:
    return abs(a - b) <= 1e-9 * max(1, abs(b))


def to_fractions(inputs: list) -> tuple:
    """Fix 1: a percentage always reaches the code as a fraction (72% -> 0.72), whichever way the model wrote it.
    Returns the normalised inputs and the names of the percentage inputs."""
    fixed, percent_names = [], set()
    for inp in inputs:
        found = NUMBER_RE.findall(inp.source.replace("$", ""))
        pct = [float(r.replace(",", "")) for r, s in found if s.lower() in ("%", "percent")]
        other = [float(r.replace(",", "")) for r, s in found if s.lower() not in ("%", "percent")]
        value = inp.value
        for n in pct:
            if _same(value, n / 100):                                     # already a fraction
                percent_names.add(inp.name)
                break
            if _same(value, n) and not any(_same(value, o) for o in other):   # written as 72 -> 0.72
                value = n / 100
                percent_names.add(inp.name)
                break
        fixed.append(inp.model_copy(update={"value": value}))
    return fixed, percent_names


# Fix 3: one worked example. It is checked by the offline tests, so it is guaranteed to pass the agent's own rules.
EXAMPLE_Q = ("A plan costs $40 per month and new customers get 25% off for their first 3 months. "
             "How much do 200 new customers pay over those 3 months?")
EXAMPLE_REPLY = {
    "status": "answer",
    "assumptions": [],
    "inputs": [{"name": "price", "value": 40, "source": "$40 per month"},
               {"name": "discount", "value": 0.25, "source": "25% off"},
               {"name": "months", "value": 3, "source": "first 3 months"},
               {"name": "customers", "value": 200, "source": "200 new customers"}],
    "code": "total = customers * price * (1 - discount) * months\nprint(round(total, 2))",
    "explanation": "Each customer pays the discounted price for 3 months.",
}
_braces = lambda s: s.replace("{", "{{").replace("}", "}}")   # keep the example's braces through .format()

AGENT_PROMPT = """You are a careful finance assistant for product managers.

Reply with ONLY a JSON object in one of these two forms.

If the question can be answered:
{{"status": "answer",
  "assumptions": ["any interpretation you had to choose, or an empty list"],
  "inputs": [{{"name": "snake_case_name", "value": <number>, "source": "<exact phrase copied from the question>"}}],
  "code": "<Python that uses ONLY the input names as variables and prints the final number>",
  "explanation": "<one sentence>"}}

If a number the calculation needs is missing, or the question has two reasonable readings that give different answers:
{{"status": "clarify", "clarifying_question": "<what you need to know>"}}

Rules:
- Every number you use must be an input whose "source" is copied word for word from the question, commas included.
  That includes conversion figures the question gives you, such as "52 weeks", "365 days" or "1 TB = 1,000 GB".
- Percentages are fractions: write 72% as 0.72. Your code receives every percentage as a fraction, so never divide one by 100.
- The code may not contain any other numbers except 0, 1, 2, 12 and 100. No imports. Use print(round(answer, 2)).
- Report a percentage answer as a percent number (15% -> 15).
- Ask only when a needed number is missing or two readings give different answers. Never ask for permission to
  proceed, and never ask to confirm a formula, method or conversion that the question already states.
  If a standard interpretation exists, answer and record it in "assumptions".

Example
Question: """ + _braces(EXAMPLE_Q) + """
""" + _braces(json.dumps(EXAMPLE_REPLY, indent=1)) + """
The numbers 40, 0.25, 3 and 200 appear only in "inputs"; the code uses their names.

Question: {question}"""

# The direct baseline gets the same clarify rule and the same example, so the comparison stays fair.
DIRECT_PROMPT = """Answer this finance question with ONLY the final number (percentages as percent numbers, e.g. 15 for 15%).
If a number you need is missing, or the question has two readings that give different answers, reply exactly: CLARIFY: <your question>
Never ask for permission to proceed, and never ask to confirm a method the question already states.

Example
Question: """ + _braces(EXAMPLE_Q) + """
18000

Question: {question}"""

def ask(question: str, model, max_attempts: int = 3) -> dict:
    """The agent: structured reply -> input checks -> sandboxed code. Errors go back to the model to fix."""
    prompt, last_error = AGENT_PROMPT.format(question=question), ""
    for attempt in range(1, max_attempts + 1):
        raw = text_of(model.invoke(prompt))
        try:
            reply = parse_reply(raw)
        except (ValidationError, ValueError) as err:
            problems = [f"invalid JSON reply: {err}"]
        else:
            if reply.status == "clarify":
                return {"status": "clarify", "clarifying_question": reply.clarifying_question, "attempts": attempt}
            problems = check_inputs(reply.inputs, question)
            if not problems:
                inputs, percent_names = to_fractions(reply.inputs)          # Fix 1
                run = run_sandboxed(reply.code, {i.name: i.value for i in inputs}, percent_names)
                if run["ok"] and first_number(run["output"]) is not None:
                    return {"status": "answer", "value": first_number(run["output"]), "output": run["output"],
                            "inputs": inputs, "assumptions": reply.assumptions, "code": reply.code,
                            "explanation": reply.explanation, "attempts": attempt}
                problems = [run["error"] or "the code printed no number"]
        last_error = "; ".join(problems)
        prompt = (f"{AGENT_PROMPT.format(question=question)}\n\nYour previous reply:\n{raw}\n\n"
                  f"It was rejected: {last_error}\nFix it and reply with ONLY the JSON object.")
    return {"status": "failed", "error": last_error, "attempts": max_attempts}


def ask_direct(question: str, model) -> dict:
    """The baseline: the LLM answers in one step, with no code and no checks."""
    raw = text_of(model.invoke(DIRECT_PROMPT.format(question=question))).strip()
    if raw.upper().startswith("CLARIFY"):
        return {"status": "clarify", "clarifying_question": raw.split(":", 1)[-1].strip()}
    value = first_number(raw)
    return {"status": "answer" if value is not None else "failed", "value": value, "assumptions": [], "output": raw}


def score(case: dict, result: dict) -> bool:
    """answer: the right number · clarify: asked for more information · clarify_or_assume: asked, or answered with a stated assumption."""
    if case["expect"] == "answer":
        if result["status"] != "answer" or result.get("value") is None:
            return False
        v, target = result["value"], case["value"]
        candidates = [v, v * 100] if case.get("unit") == "percent" else [v]
        tol = case.get("tol", max(0.01, abs(target) * 0.001))   # per-case tolerance, e.g. whole units
        return any(abs(c - target) <= tol for c in candidates)
    if case["expect"] == "clarify":
        return result["status"] == "clarify"
    return result["status"] == "clarify" or (result["status"] == "answer" and len(result.get("assumptions", [])) > 0)

def outcome(result: dict, ok: bool) -> str:
    """Fix 4: separate a confidently wrong number from a refusal. Every hard question expects a number."""
    if ok:
        return "correct"
    return {"answer": "wrong number", "clarify": "asked to clarify"}.get(result["status"], "rejected / no number")

## Step 4: Offline tests (no API key)

A scripted model stands in for the LLM, so the input checks, sandbox rules, retry loop and scoring are all tested without any API calls.

In [4]:
class Scripted:
    """Returns pre-written replies in order: an offline stand-in for an LLM."""
    def __init__(self, replies):
        self.replies, self.prompts = list(replies), []
    def invoke(self, prompt):
        self.prompts.append(prompt)
        return type("Msg", (), {"content": self.replies[min(len(self.prompts) - 1, len(self.replies) - 1)]})()

Q = "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?"
GOOD = json.dumps({"status": "answer", "assumptions": [], "explanation": "Compounded monthly.",
    "inputs": [{"name": "start", "value": 10000, "source": "starting from 10,000"},
               {"name": "rate", "value": 0.08, "source": "8% monthly"},
               {"name": "months", "value": 6, "source": "6 months"}],
    "code": "users = start * (1 + rate) ** months\nprint(round(users, 2))"})

# 1. Input provenance
I = lambda name, value, source: Input(name=name, value=value, source=source)
assert check_inputs([I("rate", 0.08, "8% monthly"), I("rate2", 8, "8% monthly")], Q) == []          # 8% may be 0.08 or 8
assert check_inputs([I("rev", 1_200_000, "$1.2M")], "Revenue grew from $1.2M to $1.5M") == []         # suffixes understood
assert "does not match" in check_inputs([I("rate", 0.8, "8% monthly")], Q)[0]                         # misread value caught
assert "does not appear" in check_inputs([I("cost", 500, "costs of 500")], Q)[0]                      # invented source caught
print("✓ 1. input checks: valid readings accepted; misread values and invented sources rejected")

# 2. Sandbox rules
assert run_sandboxed("print(round(start * (1 + rate) ** months, 2))", {"start": 10000, "rate": 0.08, "months": 6})["output"] == "15868.74"
assert "hard-coded number 10000" in run_sandboxed("print(10000 * 1.08)", {})["error"]                  # bypassing the inputs is blocked
assert "unknown variable" in run_sandboxed("print(round(price * 2, 2))", {})["error"]
assert not run_sandboxed("import os\nprint(1)", {})["ok"]
assert not run_sandboxed("print(().__class__)", {})["ok"]
print("✓ 2. sandbox: injected inputs work; hard-coded numbers, unknown names, imports and attribute tricks blocked")

# 3. The agent loop: an invented number is rejected and fixed on the second attempt
BAD = json.loads(GOOD); BAD["inputs"][1] = {"name": "rate", "value": 0.8, "source": "8% monthly"}
model = Scripted([json.dumps(BAD), GOOD])
result = ask(Q, model)
assert result["status"] == "answer" and result["value"] == 15868.74 and result["attempts"] == 2
assert "does not match any number" in model.prompts[1]                                                 # the model was told why
print("✓ 3. agent: a misread input is sent back with the reason, then corrected")

# 4. Clarification and failure paths
assert ask("What's our break-even point?", Scripted(['{"status": "clarify", "clarifying_question": "What are your costs?"}']))["status"] == "clarify"
assert ask(Q, Scripted(["not json"]))["status"] == "failed"
print("✓ 4. agent: clarifies when told to; gives up cleanly after repeated invalid replies")

# 5. Scoring
assert score({"expect": "answer", "value": 15}, {"status": "answer", "value": 15.0})
assert score({"expect": "answer", "value": 15, "unit": "percent"}, {"status": "answer", "value": 0.15})
assert not score({"expect": "answer", "value": 15}, {"status": "answer", "value": 25.0})
assert score({"expect": "clarify"}, {"status": "clarify"}) and not score({"expect": "clarify"}, {"status": "answer", "value": 1})
assert score({"expect": "clarify_or_assume"}, {"status": "answer", "value": 1, "assumptions": ["10% per month"]})
assert not score({"expect": "clarify_or_assume"}, {"status": "answer", "value": 1, "assumptions": []})
print("✓ 5. scoring: answers, clarifications and stated assumptions graded correctly")
print("\nAll offline tests passed.")

✓ 1. input checks: valid readings accepted; misread values and invented sources rejected
✓ 2. sandbox: injected inputs work; hard-coded numbers, unknown names, imports and attribute tricks blocked
✓ 3. agent: a misread input is sent back with the reason, then corrected
✓ 4. agent: clarifies when told to; gives up cleanly after repeated invalid replies
✓ 5. scoring: answers, clarifications and stated assumptions graded correctly

All offline tests passed.


### Step 4b: More Offline tests for bugs from previous runs


In [11]:
# Fix 1: percentages reach the code as fractions, whichever way the model wrote them
Q9 = "Each customer pays $49 per month, gross margin is 72%, monthly churn is 3.5%, and acquisition cost is $310. Lifetime value is monthly gross profit divided by monthly churn. What is the ratio of lifetime value to acquisition cost?"
as_written = [I("price", 49, "Each customer pays $49 per month"), I("gross_margin", 72, "gross margin is 72%"),
              I("monthly_churn", 3.5, "monthly churn is 3.5%"), I("cac", 310, "acquisition cost is $310")]
assert check_inputs(as_written, Q9) == []
inputs, pct = to_fractions(as_written)
assert [i.value for i in inputs] == [49, 0.72, 0.035, 310] and pct == {"gross_margin", "monthly_churn"}
assert to_fractions([I("rate", 0.08, "8% monthly")])[0][0].value == 0.08                     # already a fraction: unchanged
assert to_fractions([I("months", 12, "12 months at 12%")])[1] == set()                         # ambiguous 12: left alone
print("✓ fix 1a: 72% and 3.5% reach the code as 0.72 and 0.035")

# The exact bug from the first run: margin divided by 100, churn not. Now rejected, with the reason sent back.
BUGGY = {"status": "answer", "assumptions": [], "explanation": "LTV / CAC.", "inputs": [i.model_dump() for i in as_written],
         "code": "ltv = price * gross_margin / 100 / monthly_churn\nprint(round(ltv / cac, 2))"}
FIXED = dict(BUGGY, code="ltv = price * gross_margin / monthly_churn\nprint(round(ltv / cac, 2))")
model = Scripted([json.dumps(BUGGY), json.dumps(FIXED)])
result = ask(Q9, model)
assert result["status"] == "answer" and result["value"] == 3.25 and result["attempts"] == 2
assert "don't divide it by 100" in model.prompts[1]
print("✓ fix 1b: the Q9 unit bug (0.03 instead of 3.25) is caught and corrected")

# Fixes 2 and 3: the new rules are in both prompts, and the worked example passes the agent's own checks
assert "Never ask for permission" in AGENT_PROMPT and "Never ask for permission" in DIRECT_PROMPT
example = AgentReply.model_validate(EXAMPLE_REPLY)
assert check_inputs(example.inputs, EXAMPLE_Q) == []
ex_inputs, ex_pct = to_fractions(example.inputs)
assert run_sandboxed(example.code, {i.name: i.value for i in ex_inputs}, ex_pct)["output"] == "18000.0"
assert AGENT_PROMPT.format(question="x").count('"source": "$40 per month"') == 1               # braces survive .format()
print("✓ fixes 2-3: clarify rule in both prompts; the worked example is valid and computes 18000")

# Fix 4: outcome labels used by the scorecard
assert outcome({"status": "answer", "value": 1}, False) == "wrong number"
assert outcome({"status": "answer", "value": 1}, True) == "correct"
assert outcome({"status": "clarify"}, False) == "asked to clarify"
assert outcome({"status": "failed"}, False) == "rejected / no number"
print("✓ fix 4: outcomes separate wrong numbers from refusals")
print("\nAll tests passed.")

✓ fix 1a: 72% and 3.5% reach the code as 0.72 and 0.035
✓ fix 1b: the Q9 unit bug (0.03 instead of 3.25) is caught and corrected
✓ fixes 2-3: clarify rule in both prompts; the worked example is valid and computes 18000
✓ fix 4: outcomes separate wrong numbers from refusals

All tests passed.


## Step 5: Two model sizes

Check the model names against your provider's current list. Both approaches use exactly the same model and settings.

In [6]:
import os

def _load(key):
    """Colab Secrets first, then an environment variable."""
    try:
        from google.colab import userdata
        value = userdata.get(key)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(key)

PROVIDER = "openai"   # or "anthropic"
MODEL_NAMES = {
    "openai":    {"small": "gpt-5-nano", "standard": "gpt-5-mini"},
    "anthropic": {"small": "claude-haiku-4-5", "standard": "claude-sonnet-4-5"},
}
REASONING_EFFORT = "minimal"   # OpenAI GPT-5 family only; applied identically to direct and agent


def make_llm(name: str):
    if PROVIDER == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=name, api_key=_load("OPENAI_API_KEY"), reasoning_effort=REASONING_EFFORT)
    from langchain_anthropic import ChatAnthropic
    return ChatAnthropic(model=name, api_key=_load("ANTHROPIC_API_KEY"), max_tokens=1500)


MODELS = {size: make_llm(name) for size, name in MODEL_NAMES[PROVIDER].items()}
for size, name in MODEL_NAMES[PROVIDER].items():
    print(f"{size:<9} {name}")

small     gpt-5-nano
standard  gpt-5-mini


## Step 6: The harder evaluation set

| Category | Count | What makes it hard |
|---|---|---|
| `distractors` | 4 | Extra numbers (team size, salaries, launch year) that must be ignored |
| `units` | 4 | Converting weekly→monthly, TB→GB, per-day→per-quarter, daily→annual |
| `multi-step` | 8 | 4–5 chained calculations: unit economics, funnels, cash runway, pricing changes, inflation, break-even |

In [7]:
HARD_SET = [
    # --- extra numbers that aren't needed ---
    {"cat": "distractors", "q": "We have 12,400 users, 38 employees and a $2.1M budget. If 7% of users upgrade to the $29/month plan, how much monthly revenue do the upgrades add?",
     "expect": "answer", "value": 12_400 * 0.07 * 29},
    {"cat": "distractors", "q": "Our app launched in 2021 and has 4 product managers. Last quarter we had 9,500 active users, and 1,140 of them cancelled. What was the quarterly churn rate?",
     "expect": "answer", "value": 1_140 / 9_500 * 100, "unit": "percent"},
    {"cat": "distractors", "q": "A team of 6 engineers, each paid $150,000 a year, ships a feature that saves 45 support hours per week at $38 per hour. How much does it save over 52 weeks?",
     "expect": "answer", "value": 45 * 38 * 52},
    {"cat": "distractors", "q": "Plan A has 1,200 customers paying $19 a month, plan B has 450 customers paying $49, and plan C has 80 customers paying $199. What is the average monthly revenue per customer across all plans?",
     "expect": "answer", "value": (1_200 * 19 + 450 * 49 + 80 * 199) / (1_200 + 450 + 80)},
    # --- unit conversions ---
    {"cat": "units", "q": "We earn $3,250 per week from subscriptions. What is our average monthly revenue, using 52 weeks in a year?",
     "expect": "answer", "value": 3_250 * 52 / 12},
    {"cat": "units", "q": "Storage costs $0.023 per GB per month and we store 18.5 TB. What is the annual storage cost, using 1 TB = 1,000 GB?",
     "expect": "answer", "value": 0.023 * 18.5 * 1_000 * 12},
    {"cat": "units", "q": "Support handles 340 tickets per day, 7 days a week, and each ticket costs $4.75. What is the cost for one quarter, taking a quarter as 13 weeks?",
     "expect": "answer", "value": 340 * 7 * 4.75 * 13},
    {"cat": "units", "q": "Each daily active user is worth $0.18 per day. With 27,500 daily active users, what is the annual value, using 365 days?",
     "expect": "answer", "value": 0.18 * 27_500 * 365},
    # --- multi-step (4-5 calculations) ---
    {"cat": "multi-step", "q": "Each customer pays $49 per month, gross margin is 72%, monthly churn is 3.5%, and acquisition cost is $310. Lifetime value is monthly gross profit divided by monthly churn. What is the ratio of lifetime value to acquisition cost?",
     "expect": "answer", "value": (49 * 0.72 / 0.035) / 310},
    {"cat": "multi-step", "q": "Of 250,000 visitors, 4.2% start a trial and 18% of trials become paying customers at $35 per month. If new customers get 20% off their first month, what is the first-month revenue?",
     "expect": "answer", "value": 250_000 * 0.042 * 0.18 * 35 * 0.80},
    {"cat": "multi-step", "q": "We start with $1.8M in cash. Month 1 revenue is $140,000 and grows 6% each month after that, and costs are a flat $260,000 per month. How much cash is left after 6 months?",
     "expect": "answer", "value": 1_800_000 + sum(140_000 * 1.06 ** m for m in range(6)) - 6 * 260_000},
    {"cat": "multi-step", "q": "We have 8,000 customers paying $25 per month. We raise the price by 12% and expect to lose 5% of customers. By how much does monthly revenue change?",
     "expect": "answer", "value": 8_000 * 0.95 * 25 * 1.12 - 8_000 * 25},
    {"cat": "multi-step", "q": "A sales rep costs $95,000 a year. They close nothing for their first 3 months, then $22,000 of new annual contract value every month after that. Over their first 12 months, what is the contract value they close minus their annual cost?",
     "expect": "answer", "value": 22_000 * (12 - 3) - 95_000},
    {"cat": "multi-step", "q": "Revenue of $2.4M grows 15% a year for 3 years, while inflation averages 3% a year. What is year-3 revenue in today's dollars?",
     "expect": "answer", "value": 2_400_000 * 1.15 ** 3 / 1.03 ** 3},
    {"cat": "multi-step", "q": "Our conversion rate rose from 2.5% to 3.0%. By what percentage did the conversion rate increase relative to before?",
     "expect": "answer", "value": (3.0 - 2.5) / 2.5 * 100, "unit": "percent"},
    {"cat": "multi-step", "q": "Fixed costs are $84,000 a month. Each unit sells for $129 and costs $47 to make. How many units must we sell each month to break even, rounded up to a whole unit?",
     "expect": "answer", "value": 1_025, "tol": 0.5},
]

print(f"{len(HARD_SET)} questions: " + ", ".join(f"{c}={sum(x['cat'] == c for x in HARD_SET)}" for c in ["distractors", "units", "multi-step"]))
for i, c in enumerate(HARD_SET, 1):
    print(f"{i:>2}. [{c['cat']}] expected {round(c['value'], 2)}")

16 questions: distractors=4, units=4, multi-step=8
 1. [distractors] expected 25172.0
 2. [distractors] expected 12.0
 3. [distractors] expected 88920
 4. [distractors] expected 35.13
 5. [units] expected 14083.33
 6. [units] expected 5106.0
 7. [units] expected 146965.0
 8. [units] expected 1806750.0
 9. [multi-step] expected 3.25
10. [multi-step] expected 52920.0
11. [multi-step] expected 1216544.6
12. [multi-step] expected 12800.0
13. [multi-step] expected 103000
14. [multi-step] expected 3340358.57
15. [multi-step] expected 20.0
16. [multi-step] expected 1025


## Step 7: Run all four combinations

16 questions × 2 models × 2 approaches. The agent can make up to 3 attempts per question, so this takes a few minutes.

In [8]:
import time

rows = []
start = time.perf_counter()
for size, model in MODELS.items():
    for i, case in enumerate(HARD_SET, 1):
        direct = ask_direct(case["q"], model)
        agent = ask(case["q"], model)
        rows.append({"size": size, "i": i, "case": case, "direct": direct, "agent": agent,
                     "direct_ok": score(case, direct), "agent_ok": score(case, agent)})
        r = rows[-1]
        print(f"[{size:<8}] {i:>2}. direct {'✓' if r['direct_ok'] else '✗'}  agent {'✓' if r['agent_ok'] else '✗'}  {case['q'][:60]}")
print(f"\nDone in {time.perf_counter() - start:.0f} s")

[small   ]  1. direct ✗  agent ✓  We have 12,400 users, 38 employees and a $2.1M budget. If 7%
[small   ]  2. direct ✓  agent ✓  Our app launched in 2021 and has 4 product managers. Last qu
[small   ]  3. direct ✗  agent ✗  A team of 6 engineers, each paid $150,000 a year, ships a fe
[small   ]  4. direct ✗  agent ✓  Plan A has 1,200 customers paying $19 a month, plan B has 45
[small   ]  5. direct ✗  agent ✓  We earn $3,250 per week from subscriptions. What is our aver
[small   ]  6. direct ✗  agent ✗  Storage costs $0.023 per GB per month and we store 18.5 TB. 
[small   ]  7. direct ✗  agent ✓  Support handles 340 tickets per day, 7 days a week, and each
[small   ]  8. direct ✗  agent ✗  Each daily active user is worth $0.18 per day. With 27,500 d
[small   ]  9. direct ✗  agent ✗  Each customer pays $49 per month, gross margin is 72%, month
[small   ] 10. direct ✗  agent ✓  Of 250,000 visitors, 4.2% start a trial and 18% of trials be
[small   ] 11. direct ✗  agent ✗  We start with $1

## Step 8: Scorecard

Accuracy by category, then **how** each approach fails. A wrong number is the dangerous outcome: someone may act on it. A clarifying question or a rejected reply is safe.

In [9]:
def tally(size, key, cat=None):
    group = [r for r in rows if r["size"] == size and (cat is None or r["case"]["cat"] == cat)]
    return f"{sum(r[key] for r in group)}/{len(group)}"

sizes = list(MODELS)
header = "".join(f"{s + ' direct':>17}{s + ' agent':>16}" for s in sizes)
print("Correct answers")
print(f"{'':<22}{header}")
for cat in ["distractors", "units", "multi-step", None]:
    label = cat or "Overall"
    print(f"{label:<22}" + "".join(f"{tally(s, 'direct_ok', cat):>17}{tally(s, 'agent_ok', cat):>16}" for s in sizes))

# Fix 4: how each approach fails. A wrong number is the dangerous outcome; a refusal is safe.
print("\nOutcomes")
print(f"{'':<22}{header}")
for label in ["correct", "wrong number", "asked to clarify", "rejected / no number"]:
    cells = ""
    for s in sizes:
        group = [r for r in rows if r["size"] == s]
        for key, width in (("direct", 17), ("agent", 16)):
            cells += f"{sum(outcome(r[key], r[key + '_ok']) == label for r in group):>{width}}"
    print(f"{label:<22}{cells}")

print()
for s in sizes:
    retried = sum(r["agent"]["status"] == "answer" and r["agent"].get("attempts", 1) > 1 for r in rows if r["size"] == s)
    print(f"{s}: agent answers that were rejected by a check and then corrected = {retried}")

Correct answers
                           small direct     small agent  standard direct  standard agent
distractors                         1/4             3/4              1/4             4/4
units                               0/4             2/4              1/4             3/4
multi-step                          0/8             3/8              1/8             8/8
Overall                            1/16            8/16             3/16           15/16

Outcomes
                           small direct     small agent  standard direct  standard agent
correct                               1               8                3              15
wrong number                          9               1               12               0
asked to clarify                      6               7                1               0
rejected / no number                  0               0                0               1

small: agent answers that were rejected by a check and then corrected = 1
standard:

## Step 9: Where the approaches differed

For each disagreement: the expected value, both answers, and the percentage error of any wrong number. For the agent, the inputs it used are shown, so you can see *why* it got there.

In [10]:
def err(value, expected):
    if value is None:
        return ""
    return f"  ({(value - expected) / expected * 100:+.1f}%)" if expected else ""

for r in rows:
    if r["direct_ok"] == r["agent_ok"] and r["agent_ok"]:
        continue
    c, d, a = r["case"], r["direct"], r["agent"]
    print("-" * 80)
    print(f"[{r['size']}] {r['i']}. ({c['cat']}) {c['q']}")
    print(f"   expected: {round(c['value'], 2)}")
    print(f"   direct:   {d.get('value') if d['status'] == 'answer' else d['status']}{err(d.get('value'), c['value']) if not r['direct_ok'] else ''}  {'✓' if r['direct_ok'] else '✗'}")
    print(f"   outcome:  direct = {outcome(d, r['direct_ok'])}, agent = {outcome(a, r['agent_ok'])}")
    print(f"   agent:    {a.get('value') if a['status'] == 'answer' else a['status']}{err(a.get('value'), c['value']) if not r['agent_ok'] else ''}  {'✓' if r['agent_ok'] else '✗'}")
    if a["status"] == "answer":
        print("   agent inputs: " + ", ".join(f"{x.name}={x.value} ← \"{x.source}\"" for x in a["inputs"]))
    elif a["status"] == "failed":
        print(f"   agent error: {a['error'][:200]}")
    elif a["status"] == "clarify":
        print(f"   agent asked: {a['clarifying_question']}")

--------------------------------------------------------------------------------
[small] 1. (distractors) We have 12,400 users, 38 employees and a $2.1M budget. If 7% of users upgrade to the $29/month plan, how much monthly revenue do the upgrades add?
   expected: 25172.0
   direct:   114.0  (-99.5%)  ✗
   outcome:  direct = wrong number, agent = correct
   agent:    25172.0  ✓
   agent inputs: users=12400.0 ← "12,400 users", employees=38.0 ← "38 employees", budget=2100000.0 ← "$2.1M budget", upgrade_pct=0.07 ← "7% of users upgrade to the $29/month plan", upgrade_price=29.0 ← "the $29/month plan"
--------------------------------------------------------------------------------
[small] 3. (distractors) A team of 6 engineers, each paid $150,000 a year, ships a feature that saves 45 support hours per week at $38 per hour. How much does it save over 52 weeks?
   expected: 88920
   direct:   clarify  ✗
   outcome:  direct = asked to clarify, agent = wrong number
   agent:    533520.0  (+500

## How to read the results

- **Where the agent should win:** the small model answering directly on multi-step and unit questions. Arithmetic errors build up across steps, while the agent's code computes exactly.
- **Where it may not:** if the model misunderstands the question (wrong formula), code computes the wrong thing exactly. Input checks confirm a number came from the question, not that it was used correctly, so check the agent's failures for formula errors.
- **Distractors:** a wrong answer here usually means an irrelevant number was used. For the agent, the listed inputs show exactly which one.
- **Comparing with the first run:** the first run (before these fixes) had gpt-5-nano at 2/16 both ways, with **7 wrong numbers answering directly vs. 1 through the agent**, and gpt-5-mini at 3/16 direct vs. 16/16 agent. Compare the new Outcomes table with those numbers.
- **What the fixes can't do:** they help a model follow the format. If a model picks the wrong formula, code still computes the wrong thing exactly.
- **Variance:** LLM outputs vary between runs. For a firmer result, run Step 7 two or three times and compare.